First I combine the files

In [ ]:
import os
import json
import re
import unicodedata
import pandas as pd
import networkx as nx

In [ ]:
DATA_DIR = 'data'
INPUT_CSV = os.path.join(DATA_DIR, 'billboard_to_musicbrainz.csv')
GENRES_FILE = os.path.join(DATA_DIR, 'genres_FINAL.json')
OUTPUT_CSV = os.path.join(DATA_DIR, 'billboard_to_mbz_with_simplified_genres.csv')

def normalize_name(name):
    if pd.isna(name):
        return ''
    text = unicodedata.normalize('NFKD', str(name)).encode('ascii', 'ignore').decode('ascii')
    text = text.lower().strip()
    text = text.replace('&', ' and ')
    text = re.sub(r'\(.*?\)|\[.*?\]', ' ', text)
    text = re.sub(r'\b(feat|featuring|ft)\.?\b.*$', ' ', text)
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def strip_the_prefix(name):
    return re.sub(r'^the\s+', '', name).strip()

df = pd.read_csv(INPUT_CSV)

if 'Genres' in df.columns:
    df = df.drop(columns=['Genres'])

with open(GENRES_FILE, 'r', encoding='utf-8') as f:
    raw_genres = json.load(f)

normalized_genres = {}
normalized_no_the_genres = {}

for artist_key, genre_list in raw_genres.items():

    if not genre_list: 
        continue
        
    norm_name = normalize_name(artist_key)
    norm_no_the = strip_the_prefix(norm_name)

    genre_string = ", ".join(genre_list)
    
    normalized_genres[norm_name] = genre_string
    if norm_no_the:
        normalized_no_the_genres[norm_no_the] = genre_string

df['Simplified_Genres'] = pd.NA
match_count = 0

for idx, row in df.iterrows():
    listener_name = str(row['listener_artist'])
    norm = normalize_name(listener_name)
    norm_no_the = strip_the_prefix(norm)

    if norm in normalized_genres:
        df.at[idx, 'Simplified_Genres'] = normalized_genres[norm]
        match_count += 1

    elif norm_no_the in normalized_no_the_genres:
        df.at[idx, 'Simplified_Genres'] = normalized_no_the_genres[norm_no_the]
        match_count += 1
        
    else:
        main_artist_split = re.split(r'\s+and\s+|\s+&\s+', norm)
        if len(main_artist_split) > 1:
            main_artist = main_artist_split[0].strip()
            if main_artist in normalized_genres:
                df.at[idx, 'Simplified_Genres'] = normalized_genres[main_artist]
                match_count += 1

print("4. Filtering to keep ONLY matched artists...")

matched_df = df.dropna(subset=['Simplified_Genres']).copy()

matched_df.to_csv(OUTPUT_CSV, index=False)

print(f"Total artists in original Billboard CSV: {len(df)}")
print(f"Successfully matched and saved to new CSV: {len(matched_df)}")
print(f"Artists dropped (No genre found): {len(df) - len(matched_df)}")
print(f"File ready for Gephi network: {OUTPUT_CSV}")

In [ ]:
DATA_DIR = 'data'
PATH = 'genres'
os.makedirs(PATH, exist_ok=True)

NODES_CSV = os.path.join(DATA_DIR, 'billboard_to_mbz_with_simplified_genres.csv')
EDGES_CSV = os.path.join(DATA_DIR, 'mbz_feature_edges_billboard_only.csv')

STRICT_MODE = True 

nodes_df = pd.read_csv(NODES_CSV).dropna(subset=['mbid', 'Simplified_Genres'])
edges_df = pd.read_csv(EDGES_CSV).dropna(subset=['Source_MBID', 'Target_MBID'])

nodes_df['mbid'] = nodes_df['mbid'].astype(str).str.strip()
edges_df['Source_MBID'] = edges_df['Source_MBID'].astype(str).str.strip()
edges_df['Target_MBID'] = edges_df['Target_MBID'].astype(str).str.strip()

print("2. Mapping MBIDs to their Genres...")
genre_to_mbids = {}

for _, row in nodes_df.iterrows():
    mbid = row['mbid']
    genres = [g.strip() for g in str(row['Simplified_Genres']).split(',')]
    
    for genre in genres:
        if not genre: 
            continue
        if genre not in genre_to_mbids:
            genre_to_mbids[genre] = set()
        genre_to_mbids[genre].add(mbid)

sorted_genres = sorted(genre_to_mbids.items(), key=lambda item: len(item[1]), reverse=True)

print(f"\nFound {len(sorted_genres)} unique genres.")
for genre, mbid_set in sorted_genres:
    print(f"  - {genre}: {len(mbid_set)} artists")

all_genred_mbids = set(nodes_df['mbid'])

if STRICT_MODE:
    master_edges = edges_df[
        (edges_df['Source_MBID'].isin(all_genred_mbids)) & 
        (edges_df['Target_MBID'].isin(all_genred_mbids))
    ].copy()
else:
    master_edges = edges_df[edges_df['Source_MBID'].isin(all_genred_mbids)].copy()

master_output_file = os.path.join(PATH, 'network_edges_All_Genres.csv')
if len(master_edges) > 0:
    master_edges.to_csv(master_output_file, index=False)
    print(f" -> Saved [All Genres Master] Network: {len(master_edges)} edges to {master_output_file}")


for genre, mbid_set in sorted_genres:
    if len(mbid_set) < 5:
        continue

    safe_genre_name = genre.replace('&', 'n').replace(' ', '_').replace('-', '_')
    output_file = os.path.join(PATH, f'network_edges_{safe_genre_name}.csv')
    
    if STRICT_MODE:
        # Both artists must have this genre
        genre_edges = edges_df[
            (edges_df['Source_MBID'].isin(mbid_set)) & 
            (edges_df['Target_MBID'].isin(mbid_set))
        ].copy()
    else:
        # Only the main artist needs to have this genre
        genre_edges = edges_df[edges_df['Source_MBID'].isin(mbid_set)].copy()
    
    if len(genre_edges) > 0:
        genre_edges.to_csv(output_file, index=False)
        print(f" -> Saved [{genre}] Network: {len(genre_edges)} edges to {output_file}")
    else:
        print(f" -> Skipped [{genre}] Network: 0 edges found between these artists.")

Creates a network from a CSV:

In [ ]:
def build_billboard_network(top_n, 
                            nodes_file='output/billboard_to_mbz_with_simplified_genres.csv', 
                            edges_file='output_data/FINAL_network_edges.csv',
                            target_genre=None): 
    """
    Builds a self-contained NetworkX Directed Graph using the Top N Billboard artists.
    Filters nodes by a specific genre.
    """

    df_nodes = pd.read_csv(nodes_file).dropna(subset=['mbid', 'artist_mb']).copy()
    df_nodes['mbid'] = df_nodes['mbid'].astype(str).str.strip()
    df_nodes['artist_mb'] = df_nodes['artist_mb'].astype(str).str.strip()

    if target_genre:
        df_nodes = df_nodes[df_nodes['Simplified_Genres'].fillna('').str.contains(target_genre, regex=False)]

    df_nodes['chart_appearances'] = pd.to_numeric(df_nodes['chart_appearances'], errors='coerce').fillna(0)
    df_nodes = df_nodes.sort_values(by='chart_appearances', ascending=False).head(top_n)
    
    seed_mbids = set(df_nodes['mbid'])
    
    df_edges = pd.read_csv(edges_file)

    G = nx.DiGraph()
    
    # 4. Add Nodes
    for _, row in df_nodes.iterrows():
        mbid = row['mbid']
        name = row['artist_mb']
        
        G.add_node(
            mbid,
            Label=name,  
            chart_appearances=int(row['chart_appearances']),
            genres=str(row.get('Simplified_Genres', 'Unknown')),
            match_type=str(row.get('match_type', 'Unknown')),
        )

    df_edges['Source_MBID'] = df_edges['Source_MBID'].astype(str).str.strip()
    df_edges['Target_MBID'] = df_edges['Target_MBID'].astype(str).str.strip()
    
    df_self = df_edges[
        (df_edges['Source_MBID'].isin(seed_mbids)) & 
        (df_edges['Target_MBID'].isin(seed_mbids)) & 
        (df_edges['Source_MBID'] != df_edges['Target_MBID'])
    ].copy()
    
    edge_weights = (
        df_self.groupby(['Source_MBID', 'Target_MBID'], dropna=False)
        .size()
        .reset_index(name='weight')
    )
    
    for _, row in edge_weights.iterrows():
        G.add_edge(row['Source_MBID'], row['Target_MBID'], weight=int(row['weight']))
        
    return G

In [ ]:
import os

genre_files = {
    "All Genres": r"genres\network_edges_All_Genres.csv",
    "Pop": r"genres\network_edges_Pop.csv",          
    "Rock": r"genres\network_edges_Rock.csv",
    "R&B": r"genres\network_edges_RnB.csv",
    "Country": r"genres\network_edges_Country.csv",
    "Hip-Hop": r"genres\network_edges_Hip_Hop.csv",
    "Jazz": r"genres\network_edges_Jazz.csv",
    "Electronic": r"genres\network_edges_Electronic.csv"
}

MASTER_NODES_FILE = r'data\billboard_to_mbz_with_simplified_genres.csv'
genre_graphs = {}

for genre, edge_file in genre_files.items():
    print(f"\n{'='*50}")
    print(f"BUILDING {genre.upper()} NETWORK")
    print(f"{'='*50}")

    if not os.path.exists(edge_file):
        print(f"[!] Could not find {edge_file}.")
        continue


    current_target = None if genre == "All Genres" else genre

    G = build_billboard_network(
        top_n=1000000, 
        nodes_file=MASTER_NODES_FILE, 
        edges_file=edge_file,
        target_genre=current_target
    )
    
    genre_graphs[genre] = G
    print(f"Stored {genre} network in memory. Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")


Now we perform Individual Artist Analysis to find the top artists in different genres.

In [ ]:
import networkx as nx
import pandas as pd

def compile_all_network_metrics(graphs_dict):
    """
    Takes a dictionary of NetworkX graphs, calculates key centrality metrics 
    for every node, and returns a consolidated Pandas DataFrame.
    """
    all_data = []
    
    for network_name, G in graphs_dict.items():
        print(f"Processing '{network_name}' network ({G.number_of_nodes()} nodes)")
        
        if G.number_of_nodes() == 0:
            continue

        # 1. Degree Centrality
        in_degree = dict(G.in_degree())
        out_degree = dict(G.out_degree())

        # 2. Eigenvector Centrality
        try:
            eigen = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)
        except nx.PowerIterationFailedConvergence:
            print(f"  [!] Eigenvector failed to converge for {network_name}. Assigning 0s.")
            eigen = {node: 0 for node in G.nodes()}

        # 3. Betweenness Centrality
        betweenness = nx.betweenness_centrality(G)
        
        # 4. Closeness Centrality
        closeness = nx.closeness_centrality(G)

        # 5. PageRank
        pagerank = nx.pagerank(G, weight='weight')

        # 6. HITS
        try:
            hubs, authorities = nx.hits(G, max_iter=1000)
        except nx.PowerIterationFailedConvergence:
            print(f"  [!] HITS failed to converge for {network_name}. Assigning 0s.")
            hubs, authorities = {node: 0 for node in G.nodes()}, {node: 0 for node in G.nodes()}

        for node in G.nodes():
            row = {
                'Network': network_name,
                'MBID': node,
                'Artist': G.nodes[node].get('Label', node),
                'In_Degree': in_degree.get(node, 0),
                'Out_Degree': out_degree.get(node, 0),
                'Eigenvector': round(eigen.get(node, 0), 6),
                'Betweenness': round(betweenness.get(node, 0), 6),
                'Closeness': round(closeness.get(node, 0), 6),
                'PageRank': round(pagerank.get(node, 0), 6),
                'Authority_Score': round(authorities.get(node, 0), 6),
                'Hub_Score': round(hubs.get(node, 0), 6)
            }
            all_data.append(row)

    df_metrics = pd.DataFrame(all_data)
    
    return df_metrics

In [ ]:
df_master_metrics = compile_all_network_metrics(genre_graphs)

Creates a nice looking chart of all of the top artists by genre

In [ ]:
import pandas as pd

def generate_all_genres_report(df, top_n=5):

    metrics = [
        'In_Degree', 'Out_Degree', 'Closeness', 'Betweenness', 
        'Eigenvector', 'PageRank', 'Authority_Score', 'Hub_Score'
    ]
    
    all_rows = []

    for genre in sorted(df['Network'].unique()):
        sub_df = df[df['Network'] == genre]
        
        for metric in metrics:
            if metric not in sub_df.columns:
                continue

            top_sorted = sub_df.nlargest(top_n, metric)

            rank_data = []
            for _, row in top_sorted.iterrows():
                artist = str(row['Artist'])
                score = row[metric]
                
                if isinstance(score, float):
                    rank_data.append(f"{artist} ({score:.4f})")
                else:
                    rank_data.append(f"{artist} ({score})")

            while len(rank_data) < top_n:
                rank_data.append("-")

            row_dict = {
                'Genre': genre,
                'Metric': metric
            }
            for i in range(top_n):
                row_dict[f'Rank {i+1}'] = rank_data[i]
                
            all_rows.append(row_dict)

    report_df = pd.DataFrame(all_rows)

    report_df = report_df.set_index(['Genre', 'Metric'])

    styled_df = report_df.style.set_properties(**{
        'text-align': 'left',
        'white-space': 'nowrap',
        'padding': '6px 12px',
        'border': '1px solid lightgrey'
    }).set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#f0f0f0'), ('color', '#333'), ('font-weight', 'bold')]
    }])
    
    return styled_df


In [ ]:
display(generate_all_genres_report(df_master_metrics, top_n=5))

raw_report_df = generate_all_genres_report(df_master_metrics, top_n=5).data
raw_report_df.to_csv("data/ALL_GENRES_TOP_5_REPORT.csv")

Gets and Stores Gephis of all of the genres and the overall network for analysis

In [ ]:
GEPHI_DIR = 'gephi'
os.makedirs(GEPHI_DIR, exist_ok=True)


for genre, G in genre_graphs.items():

    safe_name = genre.replace('&', 'n').replace(' ', '_').replace('-', '_')
    output_path = os.path.join(GEPHI_DIR, f"{safe_name}_network.graphml")

    for node, data in G.nodes(data=True):
        for key, value in data.items():
            if value is None:
                G.nodes[node][key] = "Unknown"
            elif not isinstance(value, (int, float, str, bool)):
                G.nodes[node][key] = str(value)

    nx.write_graphml(G, output_path)
    print(f"Exported [{genre}]: {output_path}")

Now we compare between the genres:

In [ ]:
import networkx.algorithms.community as nx_comm

def compare_genre_networks(graphs_dict):

    stats_list = []
    
    for genre, G in graphs_dict.items():

        if G.number_of_nodes() == 0:
            continue

        nodes = G.number_of_nodes()
        edges = G.number_of_edges()
        

        density = nx.density(G)

        avg_degree = edges / nodes if nodes > 0 else 0

        avg_clustering = nx.average_clustering(G)

        try:
            assortativity = nx.degree_assortativity_coefficient(G)
        except Exception:
            assortativity = None
 
        G_undirected = G.to_undirected()

        try:
            communities = list(nx_comm.greedy_modularity_communities(G_undirected))
            num_communities = len(communities)
            modularity_score = nx_comm.modularity(G_undirected, communities)
        except Exception:
            num_communities = None
            modularity_score = None

        try:
            largest_cc = max(nx.connected_components(G_undirected), key=len)
            G_giant = G_undirected.subgraph(largest_cc)
            avg_path_length = nx.average_shortest_path_length(G_giant)
        except Exception:
            avg_path_length = None
            
        stats_list.append({
            'Genre': genre,
            'Total Artists (Nodes)': nodes,
            'Total Collabs (Edges)': edges,
            'Density': round(density, 6),
            'Avg Degree (Collabs/Artist)': round(avg_degree, 2),
            'Clustering Coefficient': round(avg_clustering, 4),
            'Assortativity': round(assortativity, 4) if assortativity else None,
            'Sub-Communities': num_communities,
            'Modularity Score': round(modularity_score, 4) if modularity_score else None,
            'Avg Shortest Path Length': round(avg_path_length, 4) if avg_path_length else None
        })

    df_stats = pd.DataFrame(stats_list).sort_values(by='Total Artists (Nodes)', ascending=False)

    return df_stats.set_index('Genre')



In [ ]:
genre_macro_stats = compare_genre_networks(genre_graphs)
display(genre_macro_stats)

In [10]:
genre_macro_stats.to_clipboard(sep='\t', index=True)